# Öğrenci Başarısı — Lojistik Regresyon**Veri:** Students Performance Dataset.csv (5.000 öğrenci, 23 değişken)**Hedef:** Öğrencinin dönem sonunda başarılı olup olmayacağı, dönem içi verilerle tahmin ediliyor.

## 2. Gerekli Kütüphaneler

In [ ]:
import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFoldfrom sklearn.preprocessing import StandardScalerfrom sklearn.pipeline import make_pipelinefrom sklearn.linear_model import LogisticRegressionfrom sklearn.metrics import (    accuracy_score, precision_score, recall_score, f1_score,    confusion_matrix, classification_report, roc_curve, roc_auc_score)plt.rcParams['figure.dpi'] = 150import warningswarnings.filterwarnings('ignore')

## 3. Veriyi Yükleme ve Keşfetme

In [ ]:
df = pd.read_csv("Students Performance Dataset.csv")print(f"Veri seti boyutu: {df.shape[0]} satır, {df.shape[1]} sütun")print(f"\nSütun isimleri:\n{', '.join(df.columns.tolist())}")print(f"\nİlk 5 satır:")display(df.head())print(f"\nVeri seti istatistikleri:")display(df.describe())print(f"\nEksik veri kontrolü:")print(df.isnull().sum()[df.isnull().sum() > 0].to_string())

### 3.1 Veri Seti Hakkında Kritik Bir BulguArşivde iki dosya var: `Students Performance Dataset.csv` ve `Students_Grading_Dataset_Biased.csv`.**Biased dosya kullanılmamalıdır.** O dosyada `Total_Score` ile hiçbir değişken arasında korelasyon yoktur (tüm |r| < 0.04), yani hedef değişken tamamen rastgeledir. Bu dosyada kurulan hiçbir model öğrenemez.İkinci ve daha önemli bulgu: `Total_Score` **deterministik bir formüldür.** Aşağıdaki hücrede bunu doğruluyoruz.

In [ ]:
# Total_Score gerçekten bir formül mü? Bileşen notlarıyla lineer regresyon kuralım.from sklearn.linear_model import LinearRegressionbilesenler = ['Midterm_Score', 'Final_Score', 'Assignments_Avg',              'Quizzes_Avg', 'Participation_Score', 'Projects_Score']kontrol = LinearRegression().fit(df[bilesenler], df['Total_Score'])print(f"R² = {kontrol.score(df[bilesenler], df['Total_Score']):.6f}")print("\nBulunan ağırlıklar:")for ad, w in zip(bilesenler, kontrol.coef_):    print(f"  {ad:<22}: {w:.2f}")print(f"  {'sabit terim':<22}: {kontrol.intercept_:.2f}")print("\n>>> R² = 1.000000 → Total_Score bu altı notun ağırlıklı ortalamasıdır.")print(">>> Bu altı notu birlikte özellik olarak kullanmak DAİRESEL olur:")print(">>> model tahmin etmez, sadece formülü geri çözer.")

### 3.2 Hedef Değişkenin TanımlanmasıVeri setinde hazır ikili bir hedef yok. `Grade` sütunu A–F harf notu, `Total_Score` ise sürekli puan.Lojistik regresyon için ikili hedefi kendimiz türetiyoruz:$$\text{Basari} = \begin{cases} 1 & \text{Total\_Score} \geq 70 \quad (\text{C ve üzeri}) \\ 0 & \text{Total\_Score} < 70 \quad (\text{D veya F}) \end{cases}$$

In [ ]:
# Harf notu - puan aralığı ilişkisini kontrol edelimprint("Harf notlarının puan aralıkları:")display(df.groupby('Grade')['Total_Score'].agg(['min', 'max', 'count']).round(2))# İkili hedef değişkeni oluşturdf['Basari'] = (df['Total_Score'] >= 70).astype(int)print("\nSınıf dağılımı (Basari):")print(df['Basari'].value_counts().to_string())print(f"\nOranlar: Başarılı %{df['Basari'].mean()*100:.1f} | Başarısız %{(1-df['Basari'].mean())*100:.1f}")print("\n>>> Sınıflar makul dengede, ayrıca dengeleme (SMOTE vb.) gerekmiyor.")

In [ ]:
# Sınıf dağılımı ve bazı değişkenlerin sınıflara göre dağılımıfig, ax = plt.subplots(1, 3, figsize=(15, 4))ax[0].bar(['Başarısız (0)', 'Başarılı (1)'],          df['Basari'].value_counts().sort_index().values,          color=['#e74c3c', '#2ecc71'])ax[0].set_title('Sınıf Dağılımı')ax[0].set_ylabel('Öğrenci Sayısı')for i, kolon in enumerate(['Midterm_Score', 'Study_Hours_per_Week']):    ax[i+1].hist([df[df.Basari == 0][kolon], df[df.Basari == 1][kolon]],                 bins=25, label=['Başarısız', 'Başarılı'],                 color=['#e74c3c', '#2ecc71'], alpha=0.75)    ax[i+1].set_title(f'{kolon} Dağılımı')    ax[i+1].legend()plt.tight_layout()plt.show()print("Midterm_Score iki sınıfta belirgin biçimde ayrışıyor.")print("Study_Hours_per_Week ise neredeyse tamamen üst üste — ayırt edici değil.")

## 4. Özellik Seçimi — Sızıntıdan KaçınmakBölüm 3.1'de `Total_Score`'un altı notun ağırlıklı ortalaması olduğunu gösterdik. Buradan üç farklı özellik seti tanımlıyoruz ve hangisinin **gerçek** bir tahmin problemi olduğunu tartışıyoruz:| Set | İçerik | Durum ||---|---|---|| **A** | Altı bileşen notunun tamamı | ❌ Dairesel — hedef bunlardan hesaplanıyor || **B** | Dönem içi notlar + davranışsal + demografik | ✅ Gerçek erken tahmin problemi || **C** | Sadece davranışsal/demografik | ⚠️ Referans (baseline) |**Set B'nin mantığı:** Dönem ortasındayız. `Final_Score` ve `Projects_Score` henüz belli değil (bunlar toplam notun %55'i). Elimizde vize, ödev, quiz, katılım notları ve devam/çalışma alışkanlıkları var. Soru şu: **bu bilgiyle öğrencinin dönem sonunda geçip geçmeyeceğini öngörebilir miyiz?**Ana modelimiz Set B üzerine kurulacak; A ve C karşılaştırma için Bölüm 9'da eğitilecek.

In [ ]:
# Sayısal özellikler (dönem içi + davranışsal + demografik)sayisal_ozellikler = [    'Midterm_Score', 'Assignments_Avg', 'Quizzes_Avg', 'Participation_Score',    'Attendance (%)', 'Study_Hours_per_Week', 'Sleep_Hours_per_Night',    'Stress_Level (1-10)', 'Age']# Kategorik özellikler (one-hot encoding uygulanacak)kategorik_ozellikler = [    'Gender', 'Department', 'Extracurricular_Activities',    'Internet_Access_at_Home', 'Family_Income_Level']print("Kategorik değişkenlerin seviyeleri:")for k in kategorik_ozellikler:    print(f"  {k:<30}: {df[k].unique().tolist()}")# NOT: Parent_Education_Level %20 oranında eksik olduğu için dışarıda bırakıldı.print(f"\nParent_Education_Level eksik oranı: %{df['Parent_Education_Level'].isnull().mean()*100:.1f} → kullanılmadı")

In [ ]:
# One-hot encoding ve özellik matrisinin oluşturulmasıX_kategorik = pd.get_dummies(df[kategorik_ozellikler], drop_first=True).astype(int)X = pd.concat([df[sayisal_ozellikler], X_kategorik], axis=1)y = df['Basari']print(f"Özellik matrisi (X): {X.shape}")print(f"Hedef vektör (y):   {y.shape}")print(f"\nToplam {X.shape[1]} özellik:")print(f"  {len(sayisal_ozellikler)} sayısal + {X_kategorik.shape[1]} kukla (dummy) değişken")print(f"\nÖzellikler: {X.columns.tolist()}")display(X.head())

## 5. Eğitim/Test Ayrımı

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.2, random_state=42, stratify=y)print(f"Eğitim seti: {X_train.shape[0]} örnek")print(f"Test seti:   {X_test.shape[0]} örnek")print(f"\nEğitim seti sınıf dağılımı:\n{y_train.value_counts().to_string()}")print(f"\nTest seti sınıf dağılımı:\n{y_test.value_counts().to_string()}")print("\n>>> stratify=y sayesinde sınıf oranları iki sette de korundu.")

## 6. Özellik Ölçeklendirme`fit_transform` sadece eğitim setine, `transform` test setine uygulanıyor.

In [ ]:
scaler = StandardScaler()X_train_scaled = scaler.fit_transform(X_train)X_test_scaled = scaler.transform(X_test)print("Özellikler standardize edildi (mean=0, std=1)")print(f"\nEğitim seti ilk satır (ölçeklenmiş):")print(np.round(X_train_scaled[0], 3))print(f"\nÖlçeklenmiş eğitim seti ortalamaları (≈0): {np.round(X_train_scaled.mean(axis=0)[:5], 4)}")print(f"Ölçeklenmiş eğitim seti std'leri (≈1):    {np.round(X_train_scaled.std(axis=0)[:5], 4)}")

## 7. Model Eğitimi`LogisticRegression(penalty='l2', C=1.0, solver='lbfgs', max_iter=1000)`

In [ ]:
model = LogisticRegression(    penalty='l2', C=1.0, solver='lbfgs', max_iter=1000, random_state=42)model.fit(X_train_scaled, y_train)print("Model eğitimi tamamlandı.")

In [ ]:
# Model katsayılarıprint(f"Kesişim (bias - β₀): {model.intercept_[0]:.4f}")print(f"\nKatsayılar (β₁ ... βₙ) ve odds oranları:")coef_df = pd.DataFrame({    'Özellik': X.columns,    'Katsayı': model.coef_[0],    'Odds Oranı': np.exp(model.coef_[0]),    '|Katsayı|': np.abs(model.coef_[0])}).sort_values('|Katsayı|', ascending=False)display(coef_df)

In [ ]:
# Özellik önem sıralaması görselleştirmesiplt.figure(figsize=(10, 7))colors = ['crimson' if c > 0 else 'royalblue' for c in coef_df['Katsayı']]plt.barh(coef_df['Özellik'], coef_df['Katsayı'], color=colors)plt.axvline(0, color='black', linewidth=0.8)plt.xlabel('Katsayı Değeri (standardize edilmiş)')plt.title('Lojistik Regresyon Katsayıları (Özellik Önem Sırası)')plt.gca().invert_yaxis()plt.tight_layout()plt.show()print("Kırmızı = başarı olasılığını artırıyor | Mavi = azaltıyor")

## 8. Model Değerlendirme

In [ ]:
y_pred = model.predict(X_test_scaled)y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]accuracy = accuracy_score(y_test, y_pred)precision = precision_score(y_test, y_pred)recall = recall_score(y_test, y_pred)f1 = f1_score(y_test, y_pred)auc_roc = roc_auc_score(y_test, y_pred_proba)print("=== MODEL PERFORMANS METRİKLERİ ===")print(f"  Doğruluk (Accuracy):     {accuracy:.4f}  ({(accuracy*100):.2f}%)")print(f"  Hassasiyet (Precision):  {precision:.4f}  ({(precision*100):.2f}%)")print(f"  Duyarlılık (Recall):     {recall:.4f}  ({(recall*100):.2f}%)")print(f"  F1-Skoru:                {f1:.4f}")print(f"  AUC-ROC:                 {auc_roc:.4f}")# Referans nokta: her şeye "başarılı" diyen aptal modelbaseline = y_test.mean()print(f"\n  Referans (hep '1' diyen model): {baseline:.4f}")print(f"  Modelin referansa göre kazancı: +{(accuracy - baseline)*100:.2f} puan")

In [ ]:
# Karmaşıklık Matrisicm = confusion_matrix(y_test, y_pred)print("=== KARIŞIKLIK MATRİSİ (Confusion Matrix) ===")print(f"               Tahmin: 0    Tahmin: 1")print(f"  Gerçek: 0     {cm[0,0]:3d} (TN)       {cm[0,1]:3d} (FP)")print(f"  Gerçek: 1     {cm[1,0]:3d} (FN)       {cm[1,1]:3d} (TP)")print()print("=== SINIFLANDIRMA RAPORU ===")print(classification_report(y_test, y_pred, target_names=["Başarısız", "Başarılı"]))

In [ ]:
# Karmaşıklık Matrisi ve ROC Eğrisi Görselleştirmefig, ax = plt.subplots(1, 2, figsize=(14, 5))# Confusion Matrix heatmapax[0].imshow(cm, cmap='Blues')ax[0].set_xticks([0, 1])ax[0].set_yticks([0, 1])ax[0].set_xticklabels(['Tahmin: 0', 'Tahmin: 1'])ax[0].set_yticklabels(['Gerçek: 0', 'Gerçek: 1'])for i in range(2):    for j in range(2):        etiket = "TP" if i == 1 and j == 1 else "TN" if i == 0 and j == 0 else "FP" if i == 0 and j == 1 else "FN"        ax[0].text(j, i, f'{cm[i, j]}\n({etiket})',                   ha='center', va='center', fontsize=14, fontweight='bold',                   color='white' if cm[i, j] > cm.max()/2 else 'black')ax[0].set_title('Confusion Matrix', fontsize=14)# ROC Curvefpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)ax[1].plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC (AUC = {auc_roc:.3f})')ax[1].plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random (AUC = 0.5)')ax[1].fill_between(fpr, tpr, alpha=0.2)ax[1].set_xlabel('False Positive Rate')ax[1].set_ylabel('True Positive Rate')ax[1].set_title('ROC Eğrisi', fontsize=14)ax[1].legend()ax[1].grid(alpha=0.3)plt.tight_layout()plt.show()

In [ ]:
# Tahmin edilen olasılıkların dağılımıplt.figure(figsize=(10, 5))plt.hist(y_pred_proba[y_test == 0], bins=30, alpha=0.65, label='Gerçek: Başarısız', color='#e74c3c')plt.hist(y_pred_proba[y_test == 1], bins=30, alpha=0.65, label='Gerçek: Başarılı', color='#2ecc71')plt.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Karar sınırı (0.5)')plt.xlabel('Tahmin Edilen Başarı Olasılığı')plt.ylabel('Öğrenci Sayısı')plt.title('Sigmoid Çıktılarının Dağılımı')plt.legend()plt.grid(alpha=0.3)plt.show()print("İki dağılım kısmen üst üste biniyor — model mükemmel ayıramıyor.")print("Bu beklenen bir sonuç: dönem sonu notunun %55'i (final + proje) henüz bilinmiyor.")

## 9. Özellik Setlerinin KarşılaştırılmasıŞimdi Bölüm 4'te tanımladığımız üç seti aynı koşullarda eğitip karşılaştırıyoruz. Bu bölüm, veri sızıntısının sonuçları nasıl çarpıttığını somut olarak gösteriyor.

In [ ]:
setler = {    'A: Tüm bileşen notları (SIZINTILI)': bilesenler,    'B: Dönem içi + davranışsal (ANA MODEL)': sayisal_ozellikler,    'C: Sadece davranışsal (REFERANS)': ['Attendance (%)', 'Study_Hours_per_Week',                                          'Sleep_Hours_per_Night', 'Stress_Level (1-10)', 'Age'],}cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)karsilastirma = []for ad, ozellikler in setler.items():    boru = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))    acc = cross_val_score(boru, df[ozellikler], y, cv=cv, scoring='accuracy')    auc = cross_val_score(boru, df[ozellikler], y, cv=cv, scoring='roc_auc')    karsilastirma.append({        'Özellik Seti': ad,        'Özellik Sayısı': len(ozellikler),        'CV Doğruluk': f'{acc.mean():.4f}',        'CV AUC': f'{auc.mean():.4f}'    })print("5-Katlı Çapraz Doğrulama Sonuçları:")display(pd.DataFrame(karsilastirma))

In [ ]:
# Karşılaştırmanın görselleştirilmesisonuc_df = pd.DataFrame(karsilastirma)x_pos = np.arange(len(sonuc_df))genislik = 0.35plt.figure(figsize=(11, 5))plt.bar(x_pos - genislik/2, sonuc_df['CV Doğruluk'].astype(float), genislik,        label='Doğruluk', color='#3498db')plt.bar(x_pos + genislik/2, sonuc_df['CV AUC'].astype(float), genislik,        label='AUC', color='#e67e22')plt.axhline(y=y.mean(), color='red', linestyle='--', linewidth=1.5,            label=f'Referans doğruluk ({y.mean():.3f})')plt.axhline(y=0.5, color='gray', linestyle=':', linewidth=1.5, label='Rastgele AUC (0.5)')plt.xticks(x_pos, ['A: Sızıntılı', 'B: Ana model', 'C: Referans'])plt.ylabel('Skor')plt.ylim(0, 1.1)plt.title('Özellik Setlerine Göre Model Performansı')plt.legend(loc='lower right')plt.grid(axis='y', alpha=0.3)plt.show()

### 9.1 Karşılaştırmanın Yorumu**Set A (%99+ doğruluk):** Kusursuz görünen bu sonuç aslında bir başarısızlıktır. Model hiçbir şey öğrenmemiş, sadece `Total_Score` formülünü tersine çevirmiştir. Gerçek hayatta bu özellikler elde olduğunda zaten harf notu da bellidir; tahmin edilecek bir şey kalmaz. **Ödevlerde en sık yapılan hata budur** — yüksek skor görünce durulur, skorun nereden geldiği sorgulanmaz.**Set C (AUC ≈ 0.50):** Devamsızlık, çalışma saati, uyku ve stres değişkenleri hedefi hiç açıklamıyor. AUC'nin 0.50 olması "rastgele tahmin kadar iyi" demektir. Doğruluğun yine de %59 çıkması yanıltıcıdır: model her öğrenciye "başarılı" diyerek çoğunluk sınıfının oranını yakalar. **Bu, doğruluk metriğine tek başına neden güvenilmemesi gerektiğinin klasik örneğidir.****Set B (ana model):** Doğruluk referans çizgisinin üzerinde, AUC 0.50'nin belirgin biçimde üzerinde. Model gerçekten bilgi çıkarıyor ama mükemmel değil — çünkü dönem sonu notunun büyük kısmını belirleyen final ve proje notları henüz yok. **Gerçekçi ve savunulabilir olan sonuç budur.**

## 10. Örnek Tahmin

In [ ]:
ornek = X_test.iloc[0:1]ornek_scaled = scaler.transform(ornek)ornek_olasilik = model.predict_proba(ornek_scaled)[0, 1]ornek_tahmin = model.predict(ornek_scaled)[0]print("Örnek öğrenci verisi:")display(ornek)print(f"\nBaşarı olasılığı: {ornek_olasilik:.4f} ({ornek_olasilik*100:.1f}%)")print(f"Tahmin: {ornek_tahmin} ({'Başarılı' if ornek_tahmin == 1 else 'Başarısız'})")print(f"Gerçek: {y_test.iloc[0]} ({'Başarılı' if y_test.iloc[0] == 1 else 'Başarısız'})")

In [ ]:
# Risk altındaki öğrencileri listeleme (pratik kullanım senaryosu)test_sonuclari = pd.DataFrame({    'Basari_Olasiligi': y_pred_proba,    'Tahmin': y_pred,    'Gercek': y_test.values}, index=y_test.index)riskli = test_sonuclari.sort_values('Basari_Olasiligi').head(10)riskli = riskli.join(df[['Student_ID', 'Midterm_Score', 'Attendance (%)']])print("EN RİSKLİ 10 ÖĞRENCİ (erken uyarı listesi):")display(riskli[['Student_ID', 'Midterm_Score', 'Attendance (%)',                'Basari_Olasiligi', 'Gercek']].round(3))print("\nPratik kullanım: bu öğrenciler dönem ortasında ek destek programına alınabilir.")

## 11. Bulgular**1. `Total_Score` deterministik bir formül.** Altı bileşen notunun ağırlıklı ortalaması: 0.30 proje + 0.25 final + 0.15 vize + 0.15 ödev + 0.10 quiz + 0.05 katılım. Bileşen notlarıyla kurulan lineer regresyonun R² değeri tam olarak 1.000000.**2. Üç özellik setinin karşılaştırması:**| Set | Doğruluk | AUC ||---|---|---|| A: Tüm bileşen notları | 0.996 | 1.000 || B: Dönem içi + davranışsal | 0.694 | 0.752 || C: Sadece davranışsal | 0.592 | 0.494 |Set A'nın %99.6'sı dairesel — model tahmin etmiyor, formülü geri çözüyor. Set C'nin %59 doğruluk alması ama AUC'sinin 0.49 olması, doğruluk metriğine tek başına neden güvenilmemesi gerektiğini gösteriyor: model her öğrenciye "başarılı" diyerek çoğunluk sınıfının oranını yakalıyor.**3. Ana model (Set B):** Test doğruluk 0.681, AUC 0.738. Referans (çoğunluk sınıfı) 0.592. Model gerçekten bilgi çıkarıyor ama mükemmel değil — dönem sonu notunun %55'ini belirleyen final ve proje notları henüz yok.**4. En etkili özellik:** `Midterm_Score`. Cinsiyet, bölüm, internet erişimi, aile geliri ve ders dışı etkinlik katsayıları sıfıra çok yakın — veri seti sentetik üretildiği için demografik değişkenlerle başarı arasına kasıtlı bir ilişki konulmamış.**5. Kullanılmayan dosya:** Arşivdeki `Students_Grading_Dataset_Biased.csv` dosyasında `Total_Score` ile hiçbir değişken arasında korelasyon yok (tüm |r| < 0.04). Analizde kullanılmadı.**Metodolojik çıkarım:** Model kurmadan önce hedef değişkenin nasıl üretildiğini incelemek, metrikleri iyileştirmeye çalışmaktan daha önemli.

In [ ]:
print("=== SONUÇ TABLOSU ===")print(f"  Veri seti:               {df.shape[0]} öğrenci, {X.shape[1]} özellik")print(f"  Hedef:                   Basari (Total_Score >= 70)")print(f"  Model:                   Lojistik Regresyon (L2, C=1.0)")print()print(f"  Doğruluk (Accuracy):     {accuracy:.4f}  ({(accuracy*100):.2f}%)")print(f"  Hassasiyet (Precision):  {precision:.4f}  ({(precision*100):.2f}%)")print(f"  Duyarlılık (Recall):     {recall:.4f}  ({(recall*100):.2f}%)")print(f"  F1-Skoru:                {f1:.4f}")print(f"  AUC-ROC:                 {auc_roc:.4f}")print()print(f"  Referans (çoğunluk):     {baseline:.4f}")print(f"  Sızıntılı modelin skoru: {karsilastirma[0]['CV Doğruluk']} (kullanılmadı)")